# Adım 2: Kafka ile Streaming Veri Üretimi

NYC Yellow Taxi verilerini Docker içindeki Kafka'ya akıtıyoruz.

In [ ]:
import pandas as pd
from kafka import KafkaProducer
import json
import time
import os

KAFKA_BROKER = 'kafka:9092'
TOPIC_NAME = 'taxi-topic'
DATA_PATH = '/data/yellow_tripdata_2026-01.parquet'
BATCH_SIZE = 1000
DELAY = 1

def json_serializer(data):
    return json.dumps(data).encode('utf-8')

producer = KafkaProducer(
    bootstrap_servers=[KAFKA_BROKER],
    value_serializer=json_serializer
)

print(f"Veri dosyası okunuyor: {DATA_PATH}")
df = pd.read_parquet(DATA_PATH).head(1000000)
print(f"Toplam {len(df)} kayıt okundu. Akış başlıyor...")

counter = 0
for index, row in df.iterrows():
    message = row.to_dict()
    
    message['timestamp'] = message['tpep_pickup_datetime']
    message['kullanici_id'] = f"USER_{message['VendorID']}_{index}"
    message['olay_tipi'] = "taxi_trip"
    message['ilgili_id'] = str(message['PULocationID'])
    
    for key, value in message.items():
        if hasattr(value, 'isoformat'):
            message[key] = value.isoformat()
    
    producer.send(TOPIC_NAME, value=message)
    counter += 1
    
    if counter % BATCH_SIZE == 0:
        print(f"[{time.strftime('%H:%M:%S')}] {counter} mesaj gönderildi.")
        time.sleep(DELAY)


producer.flush()

Veri dosyası okunuyor: /data/yellow_tripdata_2026-01.parquet
Toplam 1000000 kayıt okundu. Akış başlıyor...
[22:32:26] 1000 mesaj gönderildi.
[22:32:28] 2000 mesaj gönderildi.
[22:32:29] 3000 mesaj gönderildi.
[22:32:31] 4000 mesaj gönderildi.
[22:32:32] 5000 mesaj gönderildi.
[22:32:33] 6000 mesaj gönderildi.
[22:32:35] 7000 mesaj gönderildi.
[22:32:36] 8000 mesaj gönderildi.
[22:32:37] 9000 mesaj gönderildi.
[22:32:38] 10000 mesaj gönderildi.
[22:32:40] 11000 mesaj gönderildi.
[22:32:41] 12000 mesaj gönderildi.
[22:32:42] 13000 mesaj gönderildi.
[22:32:44] 14000 mesaj gönderildi.
[22:32:45] 15000 mesaj gönderildi.
[22:32:46] 16000 mesaj gönderildi.
[22:32:48] 17000 mesaj gönderildi.
[22:32:49] 18000 mesaj gönderildi.
[22:32:50] 19000 mesaj gönderildi.
[22:32:51] 20000 mesaj gönderildi.
[22:32:53] 21000 mesaj gönderildi.
[22:32:54] 22000 mesaj gönderildi.
[22:32:55] 23000 mesaj gönderildi.
[22:32:57] 24000 mesaj gönderildi.
[22:32:58] 25000 mesaj gönderildi.
[22:32:59] 26000 mesaj gönd